# Counterfactual Stability Analysis of Classical Lane Perception

**CS 131 Final Project — Lea M. Hadzic**

This notebook implements an end-to-end classical lane perception pipeline (Canny → ROI → Hough → fit → vanishing point), applies two counterfactual perturbations (synthetic shadow, motion blur), and analyzes temporal stability across two KITTI driving sequences.

## Outline

1. Setup & dependencies
2. Pipeline definitions
3. Drive 0002: clean baseline + perturbations
4. Drive 0002: stability metrics & compounding analysis
5. Drive 0026: cross-sequence validation
6. Figures for the report

## 1. Setup & dependencies

In [ ]:
import os
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

DATA_DIR = Path('data')
OUT_DIR = Path('outputs')
DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

### Consistent styling for all figures

Seaborn theme + a fixed palette mapping each condition to a color used throughout the report.

In [ ]:
sns.set_theme(
    style='whitegrid',
    context='paper',
    font_scale=1.0,
    rc={
        'figure.dpi': 110,
        'savefig.dpi': 200,
        'savefig.bbox': 'tight',
        'axes.spines.top': False,
        'axes.spines.right': False,
        'grid.alpha': 0.3,
    }
)

PALETTE = {
    'clean':  '#1f4e79',
    'shadow': '#c44e52',
    'blur':   '#5b8c5a',
}

## 2. Pipeline definitions

All lane-detection functions are defined together here so they can be applied uniformly across sequences and conditions. The pipeline is:

**RGB image → grayscale + Gaussian blur → Canny edge detection → ROI mask → probabilistic Hough → slope-based grouping → weighted least-squares lane fit → vanishing point**

In [ ]:
def detect_edges(img_rgb, blur_ksize=5, canny_lo=50, canny_hi=150):
    """Grayscale + Gaussian blur + Canny."""
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (blur_ksize, blur_ksize), 0)
    return cv2.Canny(blurred, canny_lo, canny_hi)

In [ ]:
def roi_mask(edges):
    """Trapezoidal region-of-interest mask tuned for drive_0002."""
    H, W = edges.shape
    mask = np.zeros_like(edges)
    poly = np.array([[
        (int(0.08 * W), H),
        (int(0.44 * W), int(0.45 * H)),
        (int(0.48 * W), int(0.45 * H)),
        (int(0.66 * W), H),
    ]], dtype=np.int32)
    cv2.fillPoly(mask, poly, 255)
    return cv2.bitwise_and(edges, mask), poly


def roi_mask_2(edges):
    """Re-tuned ROI for drive_0026 (different scene framing)."""
    H, W = edges.shape
    mask = np.zeros_like(edges)
    poly = np.array([[
        (int(0.00 * W), int(0.90 * H)),
        (int(0.42 * W), int(0.45 * H)),
        (int(0.55 * W), int(0.45 * H)),
        (int(0.68 * W), H),
    ]], dtype=np.int32)
    cv2.fillPoly(mask, poly, 255)
    return cv2.bitwise_and(edges, mask), poly

In [ ]:
def hough_segments(masked_edges, rho=2, theta=np.pi/180, threshold=50,
                   min_line_len=40, max_line_gap=100, min_abs_slope=0.4):
    """Probabilistic Hough line detection + slope filtering."""
    lines = cv2.HoughLinesP(masked_edges, rho, theta, threshold,
                            minLineLength=min_line_len,
                            maxLineGap=max_line_gap)
    if lines is None:
        return []
    out = []
    for x1, y1, x2, y2 in lines[:, 0]:
        if x2 == x1:
            continue
        slope = (y2 - y1) / (x2 - x1)
        if abs(slope) < min_abs_slope:
            continue
        out.append((x1, y1, x2, y2, slope))
    return out

In [ ]:
def fit_lane_line(segments):
    """Weighted least-squares line fit to segment endpoints."""
    if not segments:
        return None
    xs, ys, ws = [], [], []
    for x1, y1, x2, y2, _ in segments:
        length = np.hypot(x2 - x1, y2 - y1)
        xs.extend([x1, x2]); ys.extend([y1, y2]); ws.extend([length, length])
    m, b = np.polyfit(np.array(xs), np.array(ys), deg=1, w=np.array(ws))
    return m, b


def extrapolate(line, y_bot, y_top):
    """Endpoints of a fitted line between two y-values."""
    if line is None:
        return None
    m, b = line
    return (int((y_bot - b) / m), int(y_bot), int((y_top - b) / m), int(y_top))

In [ ]:
def _detect_lanes(img_rgb, roi_fn, y_top_frac=0.48):
    """Full lane-detection pipeline. roi_fn selects which ROI to use."""
    edges = detect_edges(img_rgb)
    masked, _ = roi_fn(edges)
    segs = hough_segments(masked)
    left  = [s for s in segs if s[4] < 0]
    right = [s for s in segs if s[4] > 0]
    left_fit  = fit_lane_line(left)
    right_fit = fit_lane_line(right)
    H = img_rgb.shape[0]
    y_bot = H - 1
    y_top = int(y_top_frac * H)
    left_seg  = extrapolate(left_fit,  y_bot, y_top)
    right_seg = extrapolate(right_fit, y_bot, y_top)
    return left_seg, right_seg, {'n_segs': len(segs), 'n_left': len(left), 'n_right': len(right)}


def detect_lanes(img_rgb, y_top_frac=0.48):
    """Lane detection using drive_0002 ROI."""
    return _detect_lanes(img_rgb, roi_mask, y_top_frac)


def detect_lanes_v2(img_rgb, y_top_frac=0.62):
    """Lane detection using drive_0026 ROI."""
    return _detect_lanes(img_rgb, roi_mask_2, y_top_frac)

In [ ]:
def vanishing_point(left_seg, right_seg):
    """Intersection of two line segments (extended infinitely)."""
    if left_seg is None or right_seg is None:
        return None
    x1, y1, x2, y2 = left_seg
    x3, y3, x4, y4 = right_seg
    denom = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
    if abs(denom) < 1e-6:
        return None
    t = ((x1 - x3) * (y3 - y4) - (y1 - y3) * (x3 - x4)) / denom
    return (x1 + t * (x2 - x1), y1 + t * (y2 - y1))


def draw_lanes(img_rgb, left_seg, right_seg, color=(0, 255, 0), thickness=4):
    """Overlay fitted lane lines on an image."""
    out = img_rgb.copy()
    for seg in (left_seg, right_seg):
        if seg is not None:
            cv2.line(out, (seg[0], seg[1]), (seg[2], seg[3]), color, thickness)
    return out

### Perturbation definitions

Two counterfactual perturbations, both deterministic so results are reproducible:

- **Synthetic shadow**: a fixed dark trapezoid (α = 0.4 multiplicative darkening) crossing the mid-distance road surface, simulating a cast shadow.
- **Motion blur**: horizontal box-filter convolution (kernel size 9 px) along the direction of ego-motion.

In [ ]:
def apply_shadow(img_rgb, shadow_poly, darkness=0.4):
    """Multiplicatively darken pixels inside a polygon to simulate a shadow."""
    out = img_rgb.copy()
    mask = np.zeros(img_rgb.shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [shadow_poly], 255)
    shadow_region = out[mask == 255].astype(np.float32) * darkness
    out[mask == 255] = shadow_region.clip(0, 255).astype(np.uint8)
    return out


def apply_motion_blur(img_rgb, kernel_size=9, angle_deg=0):
    """Directional motion blur. angle_deg=0 means horizontal (along ego-motion)."""
    k = np.zeros((kernel_size, kernel_size), dtype=np.float32)
    k[kernel_size // 2, :] = 1.0
    if angle_deg != 0:
        M = cv2.getRotationMatrix2D((kernel_size / 2, kernel_size / 2), angle_deg, 1.0)
        k = cv2.warpAffine(k, M, (kernel_size, kernel_size))
    k /= k.sum()
    return cv2.filter2D(img_rgb, -1, k)

### Helpers for batch-running the pipeline and extracting per-frame signals

In [ ]:
def download(url, dest):
    """Idempotent download."""
    if dest.exists():
        print(f'  already have {dest.name}')
        return
    print(f'  downloading {dest.name}...')
    urllib.request.urlretrieve(url, dest)
    print(f'  done ({dest.stat().st_size / 1e6:.1f} MB)')


def run_pipeline(frame_paths, transform=None, detect_fn=detect_lanes):
    """Run the full pipeline over a list of frames, optionally applying a
    perturbation transform first. Returns a list of per-frame records."""
    records = []
    for i, p in enumerate(frame_paths):
        bgr = cv2.imread(str(p))
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        if transform is not None:
            rgb = transform(rgb)
        left_seg, right_seg, _ = detect_fn(rgb)
        vp = vanishing_point(left_seg, right_seg)
        records.append({'frame': i, 'left': left_seg, 'right': right_seg, 'vp': vp})
    return records


def extract_signals(records):
    """From records, extract per-frame vp_x, vp_y, left/right slope arrays."""
    N = len(records)
    vp_x = np.full(N, np.nan); vp_y = np.full(N, np.nan)
    left_s  = np.full(N, np.nan)
    right_s = np.full(N, np.nan)
    for r in records:
        i = r['frame']
        if r['vp'] is not None:
            vp_x[i], vp_y[i] = r['vp']
        if r['left'] is not None:
            x1, y1, x2, y2 = r['left']
            if x2 != x1: left_s[i] = (y2 - y1) / (x2 - x1)
        if r['right'] is not None:
            x1, y1, x2, y2 = r['right']
            if x2 != x1: right_s[i] = (y2 - y1) / (x2 - x1)
    return vp_x, vp_y, left_s, right_s

### Stability metric definitions

In [ ]:
def m_recovery(vp_x):
    """Number of frames with a successfully recovered vanishing point."""
    return int(np.sum(~np.isnan(vp_x)))


def m_mean_dvp_x(vp_x):
    """Mean frame-to-frame |Δ vp_x| over recovered frames."""
    v = vp_x[~np.isnan(vp_x)]
    return float(np.mean(np.abs(np.diff(v))))


def m_slope_osc(left_s, right_s):
    """Average frame-to-frame |slope change| across left and right lanes."""
    dl = np.diff(left_s[~np.isnan(left_s)])
    dr = np.diff(right_s[~np.isnan(right_s)])
    return float((np.mean(np.abs(dl)) + np.mean(np.abs(dr))) / 2)


def m_vp_2d_drift(vp_x, vp_y):
    """Mean 2D Euclidean frame-to-frame VP movement."""
    valid = ~(np.isnan(vp_x) | np.isnan(vp_y))
    vx, vy = vp_x[valid], vp_y[valid]
    return float(np.mean(np.sqrt(np.diff(vx)**2 + np.diff(vy)**2)))


def m_heading_dev(vp_x_perturbed, vp_x_clean):
    """Mean per-frame deviation from clean baseline."""
    valid = ~(np.isnan(vp_x_perturbed) | np.isnan(vp_x_clean))
    return float(np.mean(np.abs(vp_x_perturbed[valid] - vp_x_clean[valid])))


def cumulative_drift_curve(vp_x):
    """Running cumulative |Δ vp_x| indexed by frame number (NaN-safe)."""
    out = np.zeros(len(vp_x))
    prev_valid, cum = None, 0.0
    for i, v in enumerate(vp_x):
        if not np.isnan(v):
            if prev_valid is not None:
                cum += abs(v - prev_valid)
            prev_valid = v
        out[i] = cum
    return out


def autocorr_abs_diff(vp_x, max_lag=15):
    """Autocorrelation of |Δ vp_x| at lags 0..max_lag."""
    d = np.abs(np.diff(vp_x[~np.isnan(vp_x)]))
    if d.std() == 0:
        return np.zeros(max_lag + 1)
    d = (d - d.mean()) / d.std()
    n = len(d)
    return np.array([np.sum(d[:n-lag] * d[lag:]) / (n - lag)
                     for lag in range(max_lag + 1)])


def compute_results(vp_x, vp_y, left_s, right_s, vp_x_clean=None):
    """Bundle all scalar metrics for one condition into a dict."""
    return {
        'recovery':    m_recovery(vp_x),
        'mean_dvp_x':  m_mean_dvp_x(vp_x),
        'slope_osc':   m_slope_osc(left_s, right_s),
        'vp_2d_drift': m_vp_2d_drift(vp_x, vp_y),
        'heading_dev': 0.0 if vp_x_clean is None else m_heading_dev(vp_x, vp_x_clean),
    }

## 3. Drive 0002: clean baseline + perturbations

Download the primary KITTI sequence, run the clean pipeline, then apply both perturbations.

In [ ]:
SEQ_NAME = '2011_09_26_drive_0002'
CALIB_DAY = '2011_09_26'
sync_zip  = DATA_DIR / f'{SEQ_NAME}_sync.zip'
calib_zip = DATA_DIR / f'{CALIB_DAY}_calib.zip'

download(f'https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/{SEQ_NAME}/{SEQ_NAME}_sync.zip', sync_zip)
download(f'https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/{CALIB_DAY}_calib.zip',         calib_zip)

for z in [sync_zip, calib_zip]:
    with zipfile.ZipFile(z) as zf:
        zf.extractall(DATA_DIR)

IMG_DIR = DATA_DIR / CALIB_DAY / f'{SEQ_NAME}_sync' / 'image_02' / 'data'
frame_paths = sorted(IMG_DIR.glob('*.png'))
print(f'drive_0002: {len(frame_paths)} frames')

### Perturbation parameters for drive 0002

In [ ]:
# Shadow polygon: tuned for drive_0002's 1242x375 image
H_full, W_full = 375, 1242
shadow_poly = np.array([
    [int(0.30 * W_full), int(0.70 * H_full)],
    [int(0.85 * W_full), int(0.55 * H_full)],
    [int(0.90 * W_full), int(0.65 * H_full)],
    [int(0.35 * W_full), int(0.80 * H_full)],
], dtype=np.int32)

### Run pipeline on all 77 frames under three conditions

In [ ]:
# Clean
records_clean = run_pipeline(frame_paths)
vp_x_clean, vp_y_clean, left_s_clean, right_s_clean = extract_signals(records_clean)
print(f'clean:  recovered {m_recovery(vp_x_clean):3d}/{len(frame_paths)}')

# Shadow
records_shadow = run_pipeline(frame_paths, transform=lambda img: apply_shadow(img, shadow_poly))
vp_x_shadow, vp_y_shadow, left_s_shadow, right_s_shadow = extract_signals(records_shadow)
print(f'shadow: recovered {m_recovery(vp_x_shadow):3d}/{len(frame_paths)}')

# Motion blur
records_blur = run_pipeline(frame_paths, transform=lambda img: apply_motion_blur(img, kernel_size=9))
vp_x_blur, vp_y_blur, left_s_blur, right_s_blur = extract_signals(records_blur)
print(f'blur:   recovered {m_recovery(vp_x_blur):3d}/{len(frame_paths)}')

# Save
np.savez(OUT_DIR / 'drive_0002_results.npz',
         vp_x_clean=vp_x_clean,   vp_y_clean=vp_y_clean,
         vp_x_shadow=vp_x_shadow, vp_y_shadow=vp_y_shadow,
         vp_x_blur=vp_x_blur,     vp_y_blur=vp_y_blur)

## 4. Drive 0002: stability metrics & compounding analysis

In [ ]:
results = {
    'clean':  compute_results(vp_x_clean,  vp_y_clean,  left_s_clean,  right_s_clean),
    'shadow': compute_results(vp_x_shadow, vp_y_shadow, left_s_shadow, right_s_shadow, vp_x_clean=vp_x_clean),
    'blur':   compute_results(vp_x_blur,   vp_y_blur,   left_s_blur,   right_s_blur,   vp_x_clean=vp_x_clean),
}

header = f"{'metric':<14s}  {'clean':>10s}  {'shadow':>10s}  {'blur':>10s}"
print(header); print('-' * len(header))
for k in ['recovery', 'mean_dvp_x', 'slope_osc', 'vp_2d_drift', 'heading_dev']:
    row = f'{k:<14s}'
    for cond in ['clean', 'shadow', 'blur']:
        row += f'  {results[cond][k]:>10.3f}'
    print(row)

## 5. Drive 0026: cross-sequence validation

Independent KITTI sequence on a different calibration day (2011-09-29). ROI re-tuned for the different scene framing; everything else identical.

In [ ]:
SEQ_NAME_2 = '2011_09_29_drive_0026'
CALIB_DAY_2 = '2011_09_29'
sync_zip_2  = DATA_DIR / f'{SEQ_NAME_2}_sync.zip'
calib_zip_2 = DATA_DIR / f'{CALIB_DAY_2}_calib.zip'

download(f'https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/{SEQ_NAME_2}/{SEQ_NAME_2}_sync.zip', sync_zip_2)
download(f'https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/{CALIB_DAY_2}_calib.zip',         calib_zip_2)

for z in [sync_zip_2, calib_zip_2]:
    with zipfile.ZipFile(z) as zf:
        zf.extractall(DATA_DIR)

IMG_DIR_2 = DATA_DIR / CALIB_DAY_2 / f'{SEQ_NAME_2}_sync' / 'image_02' / 'data'
frame_paths_2 = sorted(IMG_DIR_2.glob('*.png'))
print(f'drive_0026: {len(frame_paths_2)} frames')

In [ ]:
# Determine image dimensions and rescale shadow polygon if needed
bgr = cv2.imread(str(frame_paths_2[0]))
H2, W2 = bgr.shape[:2]
shadow_poly_2 = np.array([
    [int(0.30 * W2), int(0.70 * H2)],
    [int(0.85 * W2), int(0.55 * H2)],
    [int(0.90 * W2), int(0.65 * H2)],
    [int(0.35 * W2), int(0.80 * H2)],
], dtype=np.int32)

# Run all three conditions
records_clean_2 = run_pipeline(frame_paths_2, detect_fn=detect_lanes_v2)
vp_x_clean_2, vp_y_clean_2, left_s_clean_2, right_s_clean_2 = extract_signals(records_clean_2)
print(f'clean:  recovered {m_recovery(vp_x_clean_2):3d}/{len(frame_paths_2)}')

records_shadow_2 = run_pipeline(frame_paths_2,
                                transform=lambda img: apply_shadow(img, shadow_poly_2),
                                detect_fn=detect_lanes_v2)
vp_x_shadow_2, vp_y_shadow_2, left_s_shadow_2, right_s_shadow_2 = extract_signals(records_shadow_2)
print(f'shadow: recovered {m_recovery(vp_x_shadow_2):3d}/{len(frame_paths_2)}')

records_blur_2 = run_pipeline(frame_paths_2,
                              transform=lambda img: apply_motion_blur(img, kernel_size=9),
                              detect_fn=detect_lanes_v2)
vp_x_blur_2, vp_y_blur_2, left_s_blur_2, right_s_blur_2 = extract_signals(records_blur_2)
print(f'blur:   recovered {m_recovery(vp_x_blur_2):3d}/{len(frame_paths_2)}')

np.savez(OUT_DIR / 'drive_0026_results.npz',
         vp_x_clean=vp_x_clean_2,   vp_y_clean=vp_y_clean_2,
         vp_x_shadow=vp_x_shadow_2, vp_y_shadow=vp_y_shadow_2,
         vp_x_blur=vp_x_blur_2,     vp_y_blur=vp_y_blur_2)

In [ ]:
results_2 = {
    'clean':  compute_results(vp_x_clean_2,  vp_y_clean_2,  left_s_clean_2,  right_s_clean_2),
    'shadow': compute_results(vp_x_shadow_2, vp_y_shadow_2, left_s_shadow_2, right_s_shadow_2, vp_x_clean=vp_x_clean_2),
    'blur':   compute_results(vp_x_blur_2,   vp_y_blur_2,   left_s_blur_2,   right_s_blur_2,   vp_x_clean=vp_x_clean_2),
}

# Side-by-side comparison table
print(f"{'metric':<14s}  {'cond':<8s}  {'drive_0002':>12s}  {'drive_0026':>12s}")
print('-' * 56)
for k in ['recovery', 'mean_dvp_x', 'slope_osc', 'vp_2d_drift', 'heading_dev']:
    for cond in ['clean', 'shadow', 'blur']:
        v1 = results[cond][k]
        v2 = results_2[cond][k]
        print(f'{k:<14s}  {cond:<8s}  {v1:>12.3f}  {v2:>12.3f}')
    print()

## 6. Figures for the report

All seven figures are generated here in the order they appear in the report. Each is saved to `outputs/` as a PNG.

### Figure 1 — Perturbation examples on a single frame

In [ ]:
DEMO_IDX = 40
bgr = cv2.imread(str(frame_paths[DEMO_IDX]))
rgb_demo = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def overlay_detection(img, detect_fn=detect_lanes):
    left_seg, right_seg, _ = detect_fn(img)
    vp = vanishing_point(left_seg, right_seg)
    vis = draw_lanes(img, left_seg, right_seg)
    if vp is not None:
        cv2.circle(vis, (int(vp[0]), int(vp[1])), 8, (255, 255, 0), -1)
    return vis

panels = [
    (rgb_demo,                                   'Clean baseline'),
    (apply_shadow(rgb_demo, shadow_poly),        'With synthetic shadow ($\\alpha=0.4$)'),
    (apply_motion_blur(rgb_demo, kernel_size=9), 'With horizontal motion blur (k=9)'),
]

fig, axes = plt.subplots(3, 1, figsize=(8, 6.5))
for ax, (img, title) in zip(axes, panels):
    ax.imshow(overlay_detection(img))
    ax.set_title(title, fontsize=11, loc='left')
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig1_perturbation_examples.png')
plt.show()

### Figure 2 — Pipeline output on four consecutive frames

In [ ]:
indices = [40, 50, 60, 70]
fig, axes = plt.subplots(4, 1, figsize=(8, 8))
for ax, idx in zip(axes, indices):
    bgr = cv2.imread(str(frame_paths[idx]))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(overlay_detection(rgb))
    ax.set_title(f'frame {idx}', fontsize=10, loc='left')
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig2_consecutive_frames.png')
plt.show()

### Figure 3 — VP $x$-coordinate over time (drive 0002)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
for cond, vx in [('clean', vp_x_clean), ('shadow', vp_x_shadow), ('blur', vp_x_blur)]:
    ax.plot(vx, lw=1.5, color=PALETTE[cond], label=cond, alpha=0.9)
ax.set_xlabel('frame')
ax.set_ylabel('vanishing point $x$ (px)')
ax.set_title('VP $x$-coordinate over time under perturbation (drive 0002)',
             fontsize=11, loc='left')
ax.legend(loc='lower right', frameon=True, fontsize=9)
plt.savefig(OUT_DIR / 'fig3_vp_timeseries.png')
plt.show()

### Figure 4 — Cumulative VP drift (compounding behavior)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
for cond, vx in [('clean', vp_x_clean), ('shadow', vp_x_shadow), ('blur', vp_x_blur)]:
    cum = cumulative_drift_curve(vx)
    ax.plot(cum, lw=1.5, color=PALETTE[cond], label=cond, alpha=0.9)
ax.set_xlabel('frame')
ax.set_ylabel('cumulative $|\\Delta v_x|$ (px)')
ax.set_title('Compounding behavior: accumulated VP drift (drive 0002)',
             fontsize=11, loc='left')
ax.legend(loc='upper left', frameon=True, fontsize=9)
plt.savefig(OUT_DIR / 'fig4_cumulative_drift.png')
plt.show()

### Figure 5 — Autocorrelation of frame-to-frame instability

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
for cond, vx in [('clean', vp_x_clean), ('shadow', vp_x_shadow), ('blur', vp_x_blur)]:
    acf = autocorr_abs_diff(vx)
    ax.plot(np.arange(len(acf)), acf, 'o-',
            color=PALETTE[cond], label=cond, lw=1.5, ms=4, alpha=0.9)
ax.axhline(0, color='k', lw=0.5, alpha=0.5)
ax.set_xlabel('lag (frames)')
ax.set_ylabel('autocorr of $|\\Delta v_x|$')
ax.set_title('Temporal structure of instability (drive 0002)',
             fontsize=11, loc='left')
ax.legend(loc='upper right', frameon=True, fontsize=9)
plt.savefig(OUT_DIR / 'fig5_autocorrelation.png')
plt.show()

### Figure 6 — Cross-sequence metric comparison (drive 0002 vs drive 0026)

In [ ]:
rows = []
metric_labels = [
    ('mean_dvp_x',  'VP-x jitter (px/frame)'),
    ('slope_osc',   'Slope osc. (slope/frame)'),
    ('vp_2d_drift', '2D VP drift (px/frame)'),
    ('heading_dev', 'Heading dev. (px)'),
]
for seq_label, res in [('drive_0002', results), ('drive_0026', results_2)]:
    for cond in ['clean', 'shadow', 'blur']:
        for mkey, mlabel in metric_labels:
            rows.append({'sequence': seq_label, 'condition': cond,
                         'metric': mlabel, 'value': res[cond][mkey]})
df = pd.DataFrame(rows)

g = sns.catplot(
    data=df, kind='bar',
    x='condition', y='value',
    hue='sequence', col='metric',
    palette=['#1f4e79', '#7a4e7e'],
    col_wrap=4, height=2.8, aspect=0.95,
    sharey=False, legend=True,
)
g.set_titles('{col_name}', size=10)
g.set_axis_labels('', '')
sns.move_legend(g, 'upper center', bbox_to_anchor=(0.5, 1.05),
                ncol=2, frameon=False, title=None)
plt.savefig(OUT_DIR / 'fig6_cross_sequence_metrics.png')
plt.show()

### Figure 7 — Frame 57 deep-dive: per-frame detection succeeds while VP fails

The canonical example of the project hypothesis: lane detection looks fine in each individual frame, but the vanishing point exhibits a single-frame excursion of >200 px that is invisible to any per-frame confidence metric.

In [ ]:
indices = [56, 57, 58]
fig, axes = plt.subplots(3, 2, figsize=(11, 6.5),
                         gridspec_kw={'wspace': 0.05, 'hspace': 0.15})
for row, idx in enumerate(indices):
    bgr = cv2.imread(str(frame_paths[idx]))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    for col, (img, label) in enumerate([(rgb, 'clean'),
                                         (apply_shadow(rgb, shadow_poly), 'with shadow')]):
        axes[row, col].imshow(overlay_detection(img))
        if row == 0:
            axes[row, col].set_title(label, fontsize=11)
        if col == 0:
            axes[row, col].set_ylabel(f'frame {idx}', fontsize=10,
                                       rotation=0, labelpad=30, va='center')
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
plt.suptitle('Per-frame detection succeeds; vanishing point fails (drive 0002, shadow)',
             fontsize=11, y=1.02)
plt.savefig(OUT_DIR / 'fig7_frame57_deepdive.png')
plt.show()

In [ ]:
# Compact 2-metric chart for slide 6 (seaborn version)
rows = []
for cond in ['clean', 'shadow', 'blur']:
    for mkey, mlab in [('mean_dvp_x', 'VP-x jitter (px/frame)'),
                       ('slope_osc',  'Slope oscillation')]:
        rows.append({'condition': cond, 'metric': mlab, 'value': results[cond][mkey]})
df = pd.DataFrame(rows)

g = sns.catplot(
    data=df, kind='bar',
    x='condition', y='value',
    hue='condition', col='metric',
    palette=[PALETTE['clean'], PALETTE['shadow'], PALETTE['blur']],
    saturation=1.0,   # <-- add this
    legend=False,
    col_wrap=2, height=3.2, aspect=1.1,
    sharey=False,
)
g.set_titles('{col_name}', size=11)
g.set_axis_labels('', '')

# Add value labels on each bar
for ax in g.axes.flat:
    for p in ax.patches:
        h = p.get_height()
        ax.text(p.get_x() + p.get_width()/2, h, f'{h:.2f}',
                ha='center', va='bottom', fontsize=9)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)

plt.savefig(OUT_DIR / 'fig_slide6_two_metrics.png', dpi=200, bbox_inches='tight')
plt.show()